In [1]:
import pandas as pd

In [2]:
return_average = pd.read_csv("sac_return_total_average.csv", header = None)

In [3]:
return_average.rename(columns={0: "date", 1: "return"}, inplace=True)

In [4]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac

def newey_west_tstat(returns, maxlags=1):
    """
    논문 방식에 따른 Newey-West t-통계량 계산 함수
    입력:
        returns: 수익률 벡터 (list, np.array, pd.Series)
        maxlags: Newey-West 보정에 사용할 최대 시차
    출력:
        (평균 수익률, NW 표준오차, NW t-통계량)
    """
    returns = np.asarray(returns)
    T = len(returns)
    X = np.ones((T, 1))  # 상수항만 포함 (평균 추정)
    
    model = sm.OLS(returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    nw_cov = cov_hac(model, nlags=maxlags)
    # nw_se = np.sqrt(nw_cov[0, 0])
    # t_stat = model.params[0] / nw_se
    
    return model.params[0], model.bse[0], model.tvalues[0]


In [5]:

# 계산 실행
mean_return, nw_se, nw_tstat = newey_west_tstat(return_average["return"], maxlags=1)


In [6]:
from scipy.stats import t



# 단측 검정 (우측): P(T > t)
p_value = 1 - t.cdf(nw_tstat, df=len(return_average))

print(f"p-value = {p_value:.6f}")


p-value = 0.076885


In [7]:
nw_tstat

np.float64(1.44072488857606)

In [8]:
return_average

,date,return
0,2019-01-30,0.015650
1,2019-02-28,0.022638
2,2019-03-28,-0.005449
3,2019-04-26,0.024246
4,2019-05-24,-0.028040
...,...,...
71,2024-09-20,0.034489
72,2024-10-18,-0.000152
73,2024-11-15,-0.022596
74,2024-12-16,0.035995
